# monogpt API KEY langchain 사용

# API Key 로딩

In [ ]:
# .env
# LLM_API_KEY=your_api_key

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv(usecwd=True), override=True)

api_key = os.getenv("LLM_API_KEY")

BASE_URL = os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1")
if not os.getenv("LLM_API_KEY"):
    raise ValueError("프로젝트 .env에 LLM_API_KEY를 설정하세요.")


## OpenAI 모델

In [ ]:
llm = ChatOpenAI(
    model="gpt-5.5",
    api_key=api_key,
    base_url=BASE_URL,
    use_responses_api=False,  # base url로 할 때는 이부분 넣어야 함.
    max_tokens=526
)

In [ ]:
response = llm.invoke("파이썬 계산기 코드 작성해줘")

print(response.content)

In [ ]:
from IPython.display import display, Markdown

display(Markdown(response.content))

## Claude 모델

In [ ]:
# claude = ChatOpenAI(
#     model="claude-haiku-4.5",
#     api_key=api_key,
#     base_url=BASE_URL,
#     temperature=0,
#     max_tokens=512,
# )

claude = ChatOpenAI(
    model="claude-haiku-4.5",
    api_key=api_key,
    base_url=BASE_URL,
    use_responses_api=False,  # base url로 할 때는 이부분 넣어야 함.(MonoRouter 사용)
    max_tokens=526
)

```text
LangChain
   ↓
ChatOpenAI
   ↓
use_responses_api=False
   ↓
POST /chat/completions
   ↓
MonoRouter
   ↓
Gemini
```

In [ ]:
response = claude.invoke(
    "LangGraph와 LangChain의 차이를 설명해줘"
)

print(response.content)

In [ ]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

messages = [
    SystemMessage(
        content="당신은 생성형 AI 전문 강사입니다."
    ),
    HumanMessage(
        content="Agentic RAG를 초보자에게 설명해줘."
    )
]

response = claude.invoke(messages)

display(Markdown(response.content))

## Gemini 모델

In [ ]:
gemini = ChatOpenAI(
    model="gemini-3.5-flash",
    api_key=api_key,
    base_url=BASE_URL,
    temperature=0,
    use_responses_api=False,  # base url로 할 때는 이부분 넣어야 함.(MonoRouter 사용)
    max_tokens=2048,
    # max_tokens=512,
    reasoning_effort="minimal"
)

response = gemini.invoke("RAG를 설명해줘")

print(response.content)

# GPT / Claude / Gemini를 하나의 함수로 처리

In [ ]:
from langchain_openai import ChatOpenAI


def get_llm(
    model: str,
    temperature: float = 0,
    max_tokens: int = 512
):
    return ChatOpenAI(
        model=model,
        api_key=api_key,
        base_url=BASE_URL,
        temperature=temperature,
        use_responses_api=False,  # base url로 할 때는 이부분 넣어야 함.(MonoRouter 사용)
        max_tokens=max_tokens,
    )

## claude 모델

In [ ]:
llm = get_llm("claude-haiku-4.5")

response = llm.invoke(
    "RAG란 무엇인가?"
)

print(response.content)

## GPT 모델

In [ ]:
llm = get_llm("gpt-5.4")
response = llm.invoke(
    "RAG란 무엇인가?"
)

print(response.content)

## Gemini 모델

In [ ]:
gemini = get_llm("gemini-3.8-flash")
response = gemini.invoke(
    "RAG란 무엇인가?"
)

print(response.content)

# PromptTemplate과 Claude 연결

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 AI 교육 전문 강사입니다."
    ),
    (
        "human",
        "{question}"
    )
])

claude = get_llm("claude-haiku-4.5")

chain = prompt | claude

In [ ]:
response = chain.invoke({
    "question": "Naive RAG와 Agentic RAG의 차이를 설명해줘."
})

display(Markdown(response.content))

# LangGraph에서 사용

In [ ]:
from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END
)

claude = get_llm("claude-haiku-4.5")

In [ ]:
def chatbot(state: MessagesState):

    response = claude.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node(
    "chatbot",
    chatbot
)

builder.add_edge(
    START,
    "chatbot"
)

builder.add_edge(
    "chatbot",
    END
)

graph = builder.compile()

## Graph 시각화

In [ ]:
from IPython.display import Image, display

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

## graph 실행

In [ ]:
result = graph.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Modular RAG를 설명해줘"
        }
    ]
})

## 결과 출력

In [ ]:
display(
    Markdown(
        result["messages"][-1].content
    )
)

# Local LLM과 함께 사용

uv add langchain-openai langchain-ollama

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama


MONO_BASE_URL = os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1")
OLLAMA_BASE_URL = "http://100.91.11.54:11434"


def get_llm(
    model: str,
    provider: str = "mono",
    temperature: float = 0,
    max_tokens: int = 512
):
    
    # ── MonoGPT / OpenAI-compatible API ──
    if provider == "mono":
        return ChatOpenAI(
            model=model,
            api_key=api_key,
            base_url=MONO_BASE_URL,
            temperature=temperature,
            use_responses_api=False,
            max_tokens=max_tokens,
        )

    # ── Ollama Local LLM ──
    elif provider == "ollama":
        return ChatOllama(
            model=model,
            base_url=OLLAMA_BASE_URL,
            temperature=temperature,
            num_predict=max_tokens,
        )

    else:
        raise ValueError(
            f"지원하지 않는 provider입니다: {provider}"
        )

## MonoGPT Claude 사용

In [ ]:
llm = get_llm(
    model="claude-haiku-4.5",
    provider="mono"
)

response = llm.invoke(
    "Agentic RAG를 간단하게 설명해줘."
)

print(response.content)

## OpenAI GPT 

In [ ]:
llm = get_llm(
    model="gpt-5.4",
    provider="mono"
)


response = llm.invoke(
    "Agentic RAG를 간단하게 설명해줘."
)

print(response.content)

## Ollama EXAONE 3.5 7.8B 사용

In [ ]:
llm = get_llm(
    model="exaone3.5:7.8b",
    provider="ollama"
)

response = llm.invoke(
    "Agentic RAG를 간단하게 설명해줘."
)

print(response.content)

# prefix 방식 
## langchain 사용

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama


MONO_BASE_URL = os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1")
OLLAMA_BASE_URL = "http://100.91.11.54:11434"


def get_llm(
    model: str,
    temperature: float = 0,
    max_tokens: int = 512
):
    
    provider, model_name = model.split("/", 1)

    if provider == "mono":

        return ChatOpenAI(
            model=model_name,
            api_key=api_key,
            base_url=MONO_BASE_URL,
            temperature=temperature,
            use_responses_api=False,
            max_tokens=max_tokens,
        )

    elif provider == "ollama":

        return ChatOllama(
            model=model_name,
            base_url=OLLAMA_BASE_URL,
            temperature=temperature,
            num_predict=max_tokens,
        )

    else:
        raise ValueError(
            f"지원하지 않는 provider: {provider}"
        )

In [ ]:
claude = get_llm(
    "mono/claude-haiku-4.5"
)
response = claude.invoke(
    "Naive RAG란 무엇인가?"
)

print(response.content)

In [ ]:
gpt = get_llm(
    "mono/gpt-5.5"
)
response = gpt.invoke(
    "Naive RAG란 무엇인가?"
)

print(response.content)


In [ ]:
exaone = get_llm(
    "ollama/exaone3.5:7.8b"
)

response = exaone.invoke(
    "Naive RAG란 무엇인가?"
)

print(response.content)

## LangGraph에서 사용

In [ ]:
llm = get_llm(
    "ollama/exaone3.5:7.8b"
)

In [ ]:
llm = get_llm(
    "mono/claude-haiku-4.5"
)

In [ ]:
def chatbot(state):
    response = llm.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

In [ ]:
from IPython.display import Image, display

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)